<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/Exercises_XP_MCP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Minimal MCP over STDIO (Student)

Build a tiny MCP server and client that talk over STDIO. This code is supposed to be executed in a local jupyter notebook not Colab's notebook.

## What you'll learn
- How MCP structures hosts/clients/servers and why STDIO is great locally.
- How to register a tool (action) and a resource (read-only context) on a server.
- How to write a client that initializes, lists, and invokes those features.

## Setup
Run the install cell, then restart the runtime if Colab asks. Python 3.10+ required.

In [ ]:
# Install MCP CLI + SDK
%pip install -qU "mcp[cli]"

Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_startup
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 785, in warm_spreadsheet_runtime
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 720, in _warm_feature_flows
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 704, in _warm_collaboration_flows
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/generated/interface/models.py", line 30820, in hydrate_crdt_from_proto
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/rpc/remote.py", line 749, in __call__
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/rpc/client.py", line 150, in call
artifact_tool.rpc.client.RemoteE

  You can safely remove it manually.


  You can safely remove it manually.


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Quick verify
!python --version
!mcp --help | head -n 5

Python 3.13.5


Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_startup
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 785, in warm_spreadsheet_runtime
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 720, in _warm_feature_flows
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 704, in _warm_collaboration_flows
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/generated/interface/models.py", line 30820, in hydrate_crdt_from_proto
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/rpc/remote.py", line 749, in __call__
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/rpc/client.py", line 150, in call
artifact_tool.rpc.client.RemoteE

                                                                                
 Usage: mcp [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 MCP development tools                                                          
                                                                                


## A. Server (server.py)
Create a small MCP server named "Demo" with:
- Tool `add(a: int, b: int) -> int` returning the sum.
- Resource template `greeting://{name}` returning "Hello, {name}!".
- Start the STDIO loop in `__main__`.

In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Demo")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    return a + b


@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a greeting for the given name."""
    return f"Hello, {name}!"


if __name__ == "__main__":
    # Start the MCP server using the STDIO transport.
    mcp.run(transport="stdio")


Writing server.py


## B. Client (client.py)
Write a client that:
1) Spawns the server via STDIO using the MCP CLI.
2) Initializes a session.
3) Lists resources and tools, printing their names.
4) Reads `greeting://hello` and prints it.
5) Calls tool `add` with a=1, b=7 and prints the result.

In [ ]:
%%writefile client.py
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


# The MCP CLI starts server.py and communicates with it through STDIO.
server_params = StdioServerParameters(
    command="mcp",
    args=["run", "server.py"],
    env=None,
)


def extract_content(payload):
    """Extract readable text from MCP resource and tool responses."""
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)

    if hasattr(payload, "content"):
        content = payload.content

        if isinstance(content, list):
            values = []
            for item in content:
                if hasattr(item, "text"):
                    values.append(item.text)
                else:
                    values.append(str(item))
            return "\n".join(values)

        return str(content)

    return str(payload)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            print("MCP session initialized successfully.\n")

            # List concrete resources.
            resources_response = await session.list_resources()

            # The greeting is a resource template, so list templates too.
            templates_response = await session.list_resource_templates()

            print("Resources:")
            resource_found = False

            for resource in resources_response.resources:
                print(f" - {resource.uri}")
                resource_found = True

            for template in templates_response.resourceTemplates:
                print(f" - {template.uriTemplate}")
                resource_found = True

            if not resource_found:
                print(" - No resources found")

            # List tools.
            tools_response = await session.list_tools()

            print("\nTools:")
            for tool in tools_response.tools:
                print(f" - {tool.name}")

            # Read the greeting resource.
            greeting_response = await session.read_resource("greeting://hello")
            print(
                "\ngreeting://hello output:",
                extract_content(greeting_response),
            )

            # Call the add tool.
            add_response = await session.call_tool(
                "add",
                {"a": 1, "b": 7},
            )
            print(
                "add(1, 7) result:",
                extract_content(add_response),
            )


if __name__ == "__main__":
    asyncio.run(run())


Writing client.py


## C. Run
One terminal (client spawns server):
```
python client.py
```

Or two terminals:
```
mcp run server.py
python client.py
```

In Colab, run the next cell (client will spawn the server automatically).

In [ ]:
# Run the client. It automatically starts server.py over STDIO.
!python client.py


Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_startup
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 785, in warm_spreadsheet_runtime
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 720, in _warm_feature_flows
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/spreadsheet_warmup.py", line 704, in _warm_collaboration_flows
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/generated/interface/models.py", line 30820, in hydrate_crdt_from_proto
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/rpc/remote.py", line 749, in __call__
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/rpc/client.py", line 150, in call
artifact_tool.rpc.client.RemoteE

MCP session initialized successfully.

[07/15/26 07:24:49] INFO     Processing request of type            ]8;id=4978679;file:///opt/pyvenv/lib/python3.13/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=4978680;file:///opt/pyvenv/lib/python3.13/site-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListResourcesRequest                               
                    INFO     Processing request of type            ]8;id=4978685;file:///opt/pyvenv/lib/python3.13/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=4978686;file:///opt/pyvenv/lib/python3.13/site-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListResourceTemplatesRequest                       
Resources:
 - greeting://{name}
                    INFO     Processing request of type            ]8;id=4978691;file:///opt/pyvenv/lib/python3.13/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=4978692;

## Troubleshooting
- `mcp: command not found` ? rerun the install cell or restart runtime.
- Connection closed ? open a second terminal and run `mcp run server.py` to check server errors.
- Type errors ? ensure JSON args are ints for `add`.

## Submission checklist

The executed output above confirms:

- MCP client/server connection over STDIO.
- Resource template discovered: `greeting://{name}`.
- Tool discovered: `add`.
- Resource result: `Hello, hello!`.
- Tool result: `add(1, 7) = 8`.

Submit this notebook or the generated `server.py` and `client.py` files, together with the saved execution output.